In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
HieraCascade Full Pipeline - Final Configuration

Binary classification using Diagnosis_binary column (0=benign, 1=malignant, -1=unknown)
Compatible with your sheet.csv structure using 'Subject' column for case IDs.

Author: Student Name
Date: 2025-10-06 (Final Version)
"""



# HieraCascade: Complete Training Pipeline - FINAL

This notebook demonstrates the complete HieraCascade pipeline for **binary classification** of soft tissue tumors (malignant vs benign).

## Configuration
- **Label Column**: `Diagnosis_binary` (0=benign, 1=malignant, -1=unknown)
- **Sheet Path**: `data/sheet.csv`
- **Study ID Column**: `Subject`

## What You'll Learn
1. Load data with automatic numeric-to-text label conversion
2. Train Stage-1 model (coarse predictions + saliency)
3. Train Stage-2 model (hierarchical classification)
4. Evaluate and visualize results

## Terminal Alternative

```bash
# Quick start (both stages) - FINAL CONFIGURATION
python -m hieracascade.quick_start \
    --data_root data \
    --sheet_csv data/sheet.csv \
    --label_column Diagnosis_binary \
    --study_id_col Subject \
    --fold 0
```



## Setup and Imports



In [ ]:
import sys
import os
from pathlib import Path
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.append('..')

from hieracascade.dataio import (
    create_index_from_sheet,
    create_site_held_out_splits,
    DatasetStage1,
    DatasetStage2,
    preprocess_volume,
    FINE_TO_IDX,
    COARSE_TO_IDX
)
from hieracascade.models import build_stage1_model, build_stage2_model
from hieracascade.train_stage1 import train_stage1
from hieracascade.train_stage2 import train_stage2

# Jupyter magic
# %matplotlib inline

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")



## Configuration - FINAL

**Binary Classification Setup:**
- Uses `Diagnosis_binary` column from your CSV
- Numeric values are automatically converted:
  - `0` → `benign`
  - `1` → `malignant`
  - `-1` → `unknown`
- Study IDs from `Subject` column
- Sheet located at `data/sheet.csv`



In [ ]:
# Configuration parameters - FINAL VERSION
DATA_ROOT = '../data'
SHEET_CSV = '../data/sheet.csv'
LABEL_COLUMN = 'Diagnosis_binary'  # Binary classification (0/1/-1 → benign/malignant/unknown)
STUDY_ID_COL = 'Subject'  # Your CSV uses 'Subject' for case IDs
OUTPUT_DIR = '../outputs/hieracascade_binary'
FOLD = 0

print(f"=" * 60)
print(f"FINAL CONFIGURATION")
print(f"=" * 60)
print(f"Data root:        {DATA_ROOT}")
print(f"Sheet CSV:        {SHEET_CSV}")
print(f"Label column:     {LABEL_COLUMN} (binary classification)")
print(f"Study ID column:  {STUDY_ID_COL}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Fold:             {FOLD}")
print(f"=" * 60)



## Step 1: Load Data from sheet.csv

This step loads the labels from your CSV file with automatic conversion:
- Reads `Diagnosis_binary` column
- Converts 0 → benign, 1 → malignant, -1 → unknown
- Handles path resolution for NIfTI files
- Detects modality (CT/MRI)
- Maps to class indices



In [ ]:
print("\n" + "=" * 60)
print("STEP 1: Loading labels from sheet.csv")
print("=" * 60)

# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Load dataset index with automatic numeric-to-text conversion
labels_csv = output_dir / 'labels.csv'
index = create_index_from_sheet(
    data_root=DATA_ROOT,
    sheet_path=SHEET_CSV,
    label_column=LABEL_COLUMN,
    study_id_col=STUDY_ID_COL,
    output_csv=str(labels_csv)
)

print(f"\n✅ Loaded {len(index)} studies")
print(f"✅ Saved labels to: {labels_csv}")

# Show what labels were found
categories = sorted(set([item['category'] for item in index]))
print(f"\n✅ Categories found: {categories}")
print(f"✅ Expected: ['benign', 'malignant'] or ['benign', 'malignant', 'unknown']")



### Visualize Label Distribution



In [ ]:
# Get label distribution
categories = [item['category'] for item in index]
sites = [item['site'] for item in index]
modalities = [item['modality'] for item in index]

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Category distribution (benign vs malignant)
category_counts = pd.Series(categories).value_counts()
category_counts.plot(kind='bar', ax=axes[0], color=['lightgreen', 'salmon', 'lightgray'])
axes[0].set_title('Binary Classification Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Site distribution
pd.Series(sites).value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Site Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Site')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# Modality distribution
pd.Series(modalities).value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Modality Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Modality')
axes[2].set_ylabel('Count')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDataset Summary:")
print(f"  Total studies:    {len(index)}")
print(f"  Categories:       {len(set(categories))}")
print(f"  Sites:            {len(set(sites))}")
print(f"  CT scans:         {modalities.count('CT')}")
print(f"  MRI scans:        {modalities.count('MRI')}")
print(f"\nCategory Breakdown:")
for cat, count in pd.Series(categories).value_counts().items():
    percentage = (count / len(categories)) * 100
    print(f"  {cat:12s}: {count:3d} ({percentage:.1f}%)")



### Create Cross-Validation Splits

Uses **stratified sampling** to maintain class balance between benign and malignant cases.



In [ ]:
# Create site-held-out CV splits with stratified sampling
splits = create_site_held_out_splits(index, stratified=True)
train_index, val_index = splits[FOLD]

print(f"\n{'=' * 60}")
print(f"Fold {FOLD} Split (Site-Held-Out)")
print(f"{'=' * 60}")
print(f"Training:   {len(train_index)} studies")
print(f"Validation: {len(val_index)} studies")

# Show class distribution in train/val
train_categories = [item['category'] for item in train_index]
val_categories = [item['category'] for item in val_index]

print(f"\nTraining set distribution:")
for cat, count in pd.Series(train_categories).value_counts().items():
    percentage = (count / len(train_categories)) * 100
    print(f"  {cat:12s}: {count:3d} ({percentage:.1f}%)")

print(f"\nValidation set distribution:")
for cat, count in pd.Series(val_categories).value_counts().items():
    percentage = (count / len(val_categories)) * 100
    print(f"  {cat:12s}: {count:3d} ({percentage:.1f}%)")

# Check class balance
train_counts = pd.Series(train_categories).value_counts()
if len(train_counts) > 1:
    balance_ratio = train_counts.min() / train_counts.max()
    print(f"\n✅ Class balance ratio: {balance_ratio:.2f} (1.0 is perfect balance)")
    if balance_ratio < 0.3:
        print(f"⚠️  Warning: Significant class imbalance detected. Consider class weights.")



## Step 2: Train Stage-1 Model

Stage-1 learns:
- Binary classification (malignant vs benign)
- Saliency map generation for crop proposals

**Terminal Alternative:**
```bash
python -m hieracascade.train_stage1 \
    --config hieracascade/configs/stage1.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --output_dir outputs/stage1/fold0 \
    --fold 0
```



In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Training Stage-1 (Binary Classification + Saliency)")
print("=" * 60)

# Load Stage-1 configuration
config_path = Path('../hieracascade/configs/stage1.yaml')
with open(config_path) as f:
    config_stage1 = yaml.safe_load(f)

# Adjust config for notebook (fewer epochs for demo)
config_stage1['train']['epochs'] = 5  # Set to 20-30 for real training
config_stage1['train']['batch_size'] = 1  # Adjust based on GPU memory

# Set output directory
stage1_output = output_dir / f'stage1/fold{FOLD}'
stage1_output.mkdir(parents=True, exist_ok=True)

print(f"\nStage-1 Configuration:")
print(f"  Epochs:        {config_stage1['train']['epochs']} (set to 20+ for production)")
print(f"  Batch size:    {config_stage1['train']['batch_size']}")
print(f"  Learning rate: {config_stage1['train']['lr']}")
print(f"  Output:        {stage1_output}")
print(f"\n⚠️  Note: Training can take 10-30 minutes per epoch depending on GPU")



### Option A: Train Stage-1 (Uncomment to run)



In [ ]:
# Uncomment the following lines to train Stage-1
# print("\n🚀 Starting Stage-1 training...")
# train_stage1(
#     config=config_stage1,
#     data_root=DATA_ROOT,
#     labels_csv=str(labels_csv),
#     output_dir=str(stage1_output),
#     fold=FOLD,
#     device=device
# )
# stage1_checkpoint = str(stage1_output / 'checkpoint_best.pt')
# print(f"\n✅ Stage-1 training complete!")
# print(f"   Checkpoint: {stage1_checkpoint}")



### Option B: Load Pre-trained Stage-1 Checkpoint

If you already have a trained checkpoint, load it here:



In [ ]:
# Load pre-trained checkpoint (adjust path as needed)
stage1_checkpoint = str(stage1_output / 'checkpoint_best.pt')

if os.path.exists(stage1_checkpoint):
    print(f"✅ Found Stage-1 checkpoint: {stage1_checkpoint}")
else:
    print(f"⚠️  No checkpoint found at {stage1_checkpoint}")
    print(f"   Please train Stage-1 first or provide checkpoint path")



### Visualize Stage-1 Saliency Maps

After training, saliency maps show where the model focuses attention.



In [ ]:
# Check for saliency visualizations
viz_dir = stage1_output / 'visualizations'
if viz_dir.exists():
    saliency_files = list(viz_dir.glob('*.png'))
    if saliency_files:
        print(f"Found {len(saliency_files)} saliency visualizations")
        
        # Display first few
        n_display = min(6, len(saliency_files))
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(saliency_files[:n_display]):
            img = plt.imread(str(img_path))
            axes[idx].imshow(img)
            axes[idx].set_title(img_path.stem, fontsize=10)
            axes[idx].axis('off')
        
        # Hide unused subplots
        for idx in range(n_display, 6):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print("No saliency visualizations found yet. Train Stage-1 to generate them.")



## Step 3: Train Stage-2 Model

Stage-2 learns:
- Fine-grained binary classification
- Hierarchical predictions using crops from Stage-1 saliency

**Terminal Alternative:**
```bash
python -m hieracascade.train_stage2 \
    --config hieracascade/configs/stage2.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0 \
    --fold 0
```



In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Training Stage-2 (Hierarchical Binary Classification)")
print("=" * 60)

# Load Stage-2 configuration
config_path = Path('../hieracascade/configs/stage2.yaml')
with open(config_path) as f:
    config_stage2 = yaml.safe_load(f)

# Adjust config for notebook
config_stage2['train']['epochs'] = 5  # Set to 40-50 for real training
config_stage2['train']['batch_size'] = 1

# Set output directory
stage2_output = output_dir / f'stage2/fold{FOLD}'
stage2_output.mkdir(parents=True, exist_ok=True)

print(f"\nStage-2 Configuration:")
print(f"  Epochs:         {config_stage2['train']['epochs']} (set to 40+ for production)")
print(f"  Crops per study:{config_stage2['proposals']['K']}")
print(f"  Crop size:      {config_stage2['proposals']['crop_size']}")
print(f"  Output:         {stage2_output}")



### Train Stage-2 (Uncomment to run)



In [ ]:
# Uncomment the following lines to train Stage-2
# print("\n🚀 Starting Stage-2 training...")
# train_stage2(
#     config=config_stage2,
#     data_root=DATA_ROOT,
#     labels_csv=str(labels_csv),
#     stage1_checkpoint=stage1_checkpoint,
#     output_dir=str(stage2_output),
#     fold=FOLD,
#     device=device
# )
# stage2_checkpoint = str(stage2_output / 'checkpoint_best.pt')
# print(f"\n✅ Stage-2 training complete!")
# print(f"   Checkpoint: {stage2_checkpoint}")



## Step 4: Evaluate Model

**Terminal Command:**
```bash
python -m hieracascade.evaluate \
    --checkpoint outputs/stage2/fold0/checkpoint_best.pt \
    --stage stage2 \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0/eval \
    --fold 0
```



In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Model Evaluation")
print("=" * 60)

stage2_checkpoint = str(stage2_output / 'checkpoint_best.pt')

if os.path.exists(stage2_checkpoint):
    print(f"✅ Found Stage-2 checkpoint: {stage2_checkpoint}")
    print(f"\n📊 To evaluate, run:")
    print(f"\npython -m hieracascade.evaluate \\")
    print(f"  --checkpoint {stage2_checkpoint} \\")
    print(f"  --stage stage2 \\")
    print(f"  --data_root {DATA_ROOT} \\")
    print(f"  --labels_csv {labels_csv} \\")
    print(f"  --stage1_ckpt {stage1_checkpoint} \\")
    print(f"  --output_dir {stage2_output}/eval \\")
    print(f"  --fold {FOLD}")
else:
    print(f"⚠️  No checkpoint found at {stage2_checkpoint}")
    print(f"   Please train Stage-2 first")



### View Training Curves



In [ ]:
# Check for training curves
plots_dir = stage2_output / 'plots'
curves_path = plots_dir / 'training_curves.png'

if curves_path.exists():
    print("📈 Training curves:")
    img = plt.imread(str(curves_path))
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No training curves found yet. Train Stage-2 to generate them.")



### View Evaluation Results



In [ ]:
# Check for confusion matrix
eval_dir = stage2_output / 'eval'
cm_path = eval_dir / 'confusion_matrix_fine.png'

if cm_path.exists():
    print("📊 Confusion Matrix (Binary Classification):")
    img = plt.imread(str(cm_path))
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Load predictions CSV
    pred_csv = eval_dir / 'predictions_fine.csv'
    if pred_csv.exists():
        df_pred = pd.read_csv(pred_csv)
        print(f"\n📋 Predictions summary:")
        print(df_pred.head(10))
        
        accuracy = (df_pred['y_true'] == df_pred['y_pred']).mean()
        print(f"\n✅ Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
        
        # Calculate per-class metrics
        for category in sorted(df_pred['y_true'].unique()):
            mask = df_pred['y_true'] == category
            class_acc = (df_pred.loc[mask, 'y_true'] == df_pred.loc[mask, 'y_pred']).mean()
            print(f"   {category:12s} accuracy: {class_acc:.3f}")
else:
    print("No evaluation results found yet. Run evaluation to generate them.")



## Summary

### What We Accomplished

1. ✅ Loaded dataset from `data/sheet.csv` with `Diagnosis_binary` column
2. ✅ Automatically converted numeric labels (0/1/-1) to text (benign/malignant/unknown)
3. ✅ Created stratified cross-validation splits
4. ✅ Configured Stage-1 and Stage-2 models for binary classification
5. ✅ Set up training pipeline (ready to run)
6. ✅ Prepared evaluation workflow

### Configuration Used

```python
DATA_ROOT = '../data'
SHEET_CSV = '../data/sheet.csv'
LABEL_COLUMN = 'Diagnosis_binary'
STUDY_ID_COL = 'Subject'
```

### Next Steps

**Option 1: Run in Notebook**
- Uncomment training cells above
- Wait for training to complete (can take hours)
- Visualize results

**Option 2: Run in Terminal** (Recommended for long training)
```bash
# Quick start (both stages) - uses defaults
python -m hieracascade.quick_start \
    --data_root data \
    --fold 0 \
    --device cuda

# Or with explicit parameters
python -m hieracascade.quick_start \
    --data_root data \
    --sheet_csv data/sheet.csv \
    --label_column Diagnosis_binary \
    --study_id_col Subject \
    --output_dir outputs/binary_classification \
    --fold 0 \
    --device cuda
```

**Option 3: Train Stages Separately**
```bash
# Stage-1
python -m hieracascade.train_stage1 \
    --config hieracascade/configs/stage1.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --output_dir outputs/stage1/fold0 \
    --fold 0

# Stage-2  
python -m hieracascade.train_stage2 \
    --config hieracascade/configs/stage2.yaml \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0 \
    --fold 0

# Evaluate
python -m hieracascade.evaluate \
    --checkpoint outputs/stage2/fold0/checkpoint_best.pt \
    --stage stage2 \
    --data_root data \
    --labels_csv outputs/hieracascade_binary/labels.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0/eval \
    --fold 0
```

### Documentation

- **FINAL_CONFIGURATION.md** - Complete setup guide
- **CHANGES_SUMMARY.md** - What changed in this version
- **hieracascade/LABEL_SELECTION_GUIDE.md** - Label column options
- **hieracascade/README.md** - General documentation
- **hieracascade/TUTORIAL.md** - Detailed tutorial



In [ ]:
print("\n" + "=" * 60)
print("✅ Pipeline Setup Complete!")
print("=" * 60)
print(f"\n📂 All outputs will be saved to: {OUTPUT_DIR}")
print(f"\n🎯 Configuration:")
print(f"   - Binary classification: benign vs malignant")
print(f"   - Data: {SHEET_CSV}")
print(f"   - Labels auto-converted from numeric (0/1/-1)")
print(f"\n💡 Next: Uncomment training cells above or use terminal commands")
print(f"   Minimal: python -m hieracascade.quick_start --data_root data --fold 0")
print("\n" + "=" * 60)

